# Multi-agent SAC for MPE and more 

* MASAC: Multi-Agent Soft Actor-Critic for Mixed Cooperative-Competitive Environments (no papers found)
* MPE: Multi-Agent Particle Environment using pettingzoo library [github](https://github.com/Farama-Foundation/PettingZoo)
* SAC: Soft Actor-Critic (2018) [arxiv](https://arxiv.org/abs/1812.05905)

## MASAC Algorithm 

**MASAC**(Multi-Agent Soft Actor-Critic) extends the maximum entropy reinforcement learning approach of SAC to multi-agent environments, combining SAC's stochastic policies and entropy regularization with MADDPG's centralized critic and decentralized actor training framework.

#### Key Components 

MASAC incorporates the following distinctive features:

* **Twin Centralized Critics**: Each agent maintains two soft centralized Q-networks that incorporate joint observations and actions, plus an entropy bonus to reduce overestimation bias (similar to MATD3).
* **Stochastic Policies**: Unlike MADDPG/MATD3's deterministic policies, MASAC uses stochastic policies that output a Gaussian distribution over actions (with a mean and standard deviation).
* **Entropy Regularization**: MASAC adds an entropy term to the policy loss to encourage exploration. 
* **Temperature Parameter**: An adjustable entropy weight $\alpha$ balances exploration and exploitation.
* **Automatic Entropy Tuning**: Optional (often beneficial) mechanism to automatically adjust the temperature parameter $\alpha$ to achieve a target entropy level.

#### Twin Critics Update 

For each agent i, maintain two soft Q-functions $Q_{\theta_{i,1}}$ and $Q_{\theta_{i,2}}$:

$$ 
\begin{aligned} 
\mathcal{L}(\theta_{i,k}) &= \mathbb{E}_{\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}'}[ (Q_{\theta_{i,k}} (\vec{o}, a_1, \dots, a_N) - y_i)^2 ] & \text{for } k \in {1,2} \\

\text{where } y_i &= r_i + \gamma \mathbb{E}_{a'_j \sim \pi_j(o'_j)} \left[ \min_{k=1,2} Q_{\overline{\theta}_{i,k}} (\vec{o}', a'_1, \dots, a'_N) - \alpha_i \log \pi_i(a'_i|o'_i) \right] 
\end{aligned} 
$$

,where $\vec{\pi}$ are the policies and $Q_{\overline{\theta}_{i,k}}$ target Q-functions with softly-updated parameters and $Q_{\theta_{i,k}}$
online Q-functions with current parameters.

#### Policy Update 

Unline MADDPG/MATD3 deterministic policy gradients, MASAC uses the reparameterization trick to sample actions from a Gaussian policy:

$$ 
\nabla_{\theta_i} J({\pi_i}) = \mathbb{E}_{\vec{o} \sim \mathcal{D},\epsilon \sim \mathcal{N}} \left[ \nabla_{\theta_i} \left( \alpha_i \log \pi_i(a_i | o_i) - \min_{k=1,2} Q_{\theta_{i,k}}(\vec{o}, a_1, \dots,a_i, \dots, a_N) \right) \right] \ \text{where } a_i = f_{\theta_i}(\epsilon_i; o_i) 
$$

Where 
* $\mathcal{D}$ is the memory buffer for experience replay, containing multiple episode samples $(\vec{o}, a_1, \dots, a_N, r_1, \dots, r_N, \vec{o}’)$
*  $f_{\theta_i}(\epsilon_i; o_i)$ is the reparameterized action using the policy network and random noise $\epsilon_i$ sampled from a standard normal distribution. 

$$f_{\theta_i}(\epsilon_i; o_i)= \mu_{\theta_i}(o_i) + \sigma_{\theta_i}(o_i) \cdot \epsilon_i, \epsilon_i \sim \mathcal{N}(0, 1)$$

$$\nabla_{\theta_i} a_i = \nabla_{\theta_i} f_{\theta_i}(\epsilon_i; o_i)$$

#### Automatic Entropy Tuning

SAC is brittle with respect to the temperature parameter. Unfortunately it is difficult to adjust temperature, because the entropy can vary unpredictably both across tasks and during training as the policy becomes better. An improvement on SAC formulates a constrained optimization problem, while maximizing the expected return, the policy should satisfy a minimum entropy constraint. The optimization problem is:
$$ J(\alpha_i) = \mathbb{E}_{a_i \sim \pi_i, o_i \sim \mathcal{D}} \left[ -\alpha_i \log \pi_i(a_i|o_i) - \alpha_i \mathcal{H}_{\text{target}} \right] $$
$$ \nabla_{\alpha_i} J(\alpha_i) = \mathbb{E}_{a_i \sim \pi_i, o_i \sim \mathcal{D}} \left[ - \log \pi_i(a_i|o_i) - \mathcal{H}_{\text{target}} \right] $$

where $\mathcal{H}_{\text{target}}$ is a target entropy value, typically set to $-\dim(\mathcal{A}_i)$.

#### Advantages over MADDPG and MATD3
1. **Improved Exploration**: Entropy maximization encourages exploration in uncertain regions of the state space
2. **Better Performance in Multi-modal Tasks**: Stochastic policies can represent multiple good strategies
3. **Reduced Sensitivity to Hyperparameters**: Automatic entropy tuning adapts exploration to each environment
4. **Robustness to Non-stationarity**: Entropy-regularized policies are more robust to changes in other agents' behaviors
5. **Sample Efficiency**: Typically requires fewer environment interactions than MADDPG to reach comparable performance

MASAC combines the strengths of maximum entropy reinforcement learning with multi-agent coordination, making it particularly effective for complex collaborative tasks requiring both exploration and precise coordination.

### MPE Environment

Great! 🎉 🎉 🎉  We done with theorectical part for MASAC! 🥳 

Now let's implement MASAC on MPE environment. MPE is a simple multi-agent environment where agents must learn to coordinate to solve tasks.

In [ ]:
# TODO: 
# 1. Define Env 
# 2. Define Model 
# 3. Define Buffer
# 4. Define SAC Agent
# 5. Define MASAC Agent 